# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
"I chose Random Forest classifier because my Week 1–4 work already showed the relationship between signals and decline isn't simple linear separation. The position alone barely separated classes in the Week 2 work, and staleness turned out unreliable in this slice in Week 4"

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [5]:
%pip -q install duckdb huggingface_hub
import duckdb
import pandas as pd
from google.colab import userdata
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DEV_MONTH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
DIM_CLIENTS = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

#Initializing Labels for Signals
labels = con.sql(f"""
    SELECT content_hash_id,
      AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks END) AS clicks_first_half,
      AVG(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_clicks END) AS clicks_second_half
    FROM read_parquet('{DEV_MONTH_PATH}')
    GROUP BY content_hash_id
""").df()
labels['is_declining'] = (labels['clicks_second_half'] < labels['clicks_first_half']).astype(int)

# Rebuild the feature frame with client_hash_id for grouping
model_df = con.sql(f"""
    SELECT
      f.content_hash_id, f.client_hash_id,
      AVG(f.gsc_avg_position) AS avg_position,
      SUM(f.gsc_clicks) AS clicks_total,
      SUM(f.gsc_impressions) AS impressions_total,
      MAX(f.report_date) AS last_report_date
    FROM read_parquet('{DEV_MONTH_PATH}') f
    GROUP BY f.content_hash_id, f.client_hash_id
""").df()

dim = con.sql(f"""
    SELECT content_hash_id, content_type, word_count, main_intent,
           content_created_date
    FROM read_parquet('{DIM_CONTENT}')
""").df()

model_df = model_df.merge(dim, on='content_hash_id').merge(
    labels[['content_hash_id','is_declining']], on='content_hash_id'
)
model_df['content_age_days'] = (
    pd.to_datetime(model_df['last_report_date']) - pd.to_datetime(model_df['content_created_date'])
).dt.days
model_df['ctr'] = model_df['clicks_total'] / model_df['impressions_total'].replace(0, pd.NA)
model_df = model_df.dropna(subset=['avg_position','word_count','content_age_days','ctr','client_hash_id'])

from sklearn.model_selection import GroupShuffleSplit

# GROUPED split: split by client_hash_id, not by row, so the same client
# never appears in both train and test — otherwise the model could learn
# client-specific quirks rather than generalizable signal.
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]

print("Train clients:", train_df['client_hash_id'].nunique(), "| Test clients:", test_df['client_hash_id'].nunique())
print("Overlap (should be 0):", len(set(train_df['client_hash_id']) & set(test_df['client_hash_id'])))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train clients: 32 | Test clients: 15
Overlap (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score

num_features = ['avg_position','word_count','content_age_days','impressions_total']
cat_features = ['content_type','main_intent']

preproc = ColumnTransformer([
    ('num', 'passthrough', num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

rf = Pipeline([
    ('prep', preproc),
    ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=0, n_jobs=-1))
])

X_tr, y_tr = train_df[num_features + cat_features], train_df['is_declining']
X_te, y_te = test_df[num_features + cat_features], test_df['is_declining']

rf.fit(X_tr, y_tr)
rf_probs = rf.predict_proba(X_te)[:,1]
rf_auc = roc_auc_score(y_te, rf_probs)
rf_f1 = f1_score(y_te, rf.predict(X_te))

# Baseline 1: Week 3 logistic regression, same grouped split for fair comparison
from sklearn.linear_model import LogisticRegression
logit = Pipeline([('prep', preproc), ('clf', LogisticRegression(max_iter=1000))])
logit.fit(X_tr, y_tr)
logit_auc = roc_auc_score(y_te, logit.predict_proba(X_te)[:,1])
logit_f1 = f1_score(y_te, logit.predict(X_te))

# Baseline 2: Week 4 rule (CTR < threshold at position 4-20) as a hard classifier
rule_pred = ((test_df['avg_position'].between(4,20)) & (test_df['ctr'] < 0.0065)).astype(int)
rule_f1 = f1_score(y_te, rule_pred)

import pandas as pd
comparison = pd.DataFrame({
    'method': ['Week 4 rule (CTR<0.0065)', 'Week 3 Logistic Regression', 'Random Forest'],
    'AUC': [None, logit_auc, rf_auc],
    'F1': [rule_f1, logit_f1, rf_f1]
})
print(comparison)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                       method       AUC        F1
0    Week 4 rule (CTR<0.0065)       NaN  0.186173
1  Week 3 Logistic Regression  0.658889  0.028006
2               Random Forest  0.848108  0.207675


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_te, y_te, n_repeats=10, random_state=0, n_jobs=-1)
importances = pd.Series(perm.importances_mean, index=num_features + cat_features).sort_values(ascending=False)
print(importances)

test_df = test_df.copy()
test_df['rf_pred'] = rf.predict(X_te)
false_negs = test_df[(test_df['is_declining']==1) & (test_df['rf_pred']==0)]
print("False negative rate:", len(false_negs) / (test_df['is_declining']==1).sum())
false_negs[['content_hash_id','avg_position','ctr','word_count','content_age_days']].describe()

#NOTE
"The Random Forest leans almost entirely on impressions_total. This produced a high false negative rate (87.4%), meaning the model systematically misses declining pages that have moderate position and moderate age"

impressions_total    0.024360
content_age_days     0.007657
content_type         0.000745
main_intent          0.000166
word_count           0.000109
avg_position        -0.000770
dtype: float64
False negative rate: 0.8741293532338309


'The Random Forest leans almost entirely on impressions_total. This produced a high false negative rate (87.4%), meaning the model systematically misses declining pages that have moderate position and moderate age'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.